# Credit Default Prediction

In [ ]:
import urllib.request
import json
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from IPython import display
from sklearn.metrics import precision_score, recall_score, f1_score

# --- Configuration ---
URL = '<your-endpoint-url>'
API_KEY = '<your-api-key>'
DATA_PATH = '../credit_default/dataset/credit_default_2.csv'
TOTAL_DURATION = 100 
WINDOW_SIZE = 100 # Number of recent samples for the "rolling" calculation
SAMPLES_PER_UPDATE = 10 # Number of samples per endpoint request

# --- Load and Prepare Data ---
df = pd.read_csv(DATA_PATH)
X_all = df.drop(['DEFAULT'], axis=1)
y_all = df['DEFAULT']

mask_drift = df['LIMIT_BAL'] >= 150000
X_normal, y_normal = X_all[~mask_drift], y_all[~mask_drift]
X_drift, y_drift = X_all[mask_drift], y_all[mask_drift]

# --- Metrics Tracking ---
history = []      # Raw results: {'actual': y, 'pred': y_p}
plot_times = []
plot_precision = []
plot_recall = []
plot_f1 = []
plot_drift_pct = []

def invoke_endpoint(sample_rows):
    # Matches the required format: {"data": [[val1, val2, ...]]}
    payload = {"data": sample_rows.to_numpy().tolist()}
    body = str.encode(json.dumps(payload))
    headers = {'Content-Type': 'application/json', 'Authorization': f'Bearer {API_KEY}'}
    
    req = urllib.request.Request(URL, body, headers)
    try:
        with urllib.request.urlopen(req) as response:
            res = json.loads(response.read().decode("utf8"))
            # Extracts prediction from list or dict response
            return res if isinstance(res, list) else res.get('predict', [None])
    except Exception:
        return None

# --- Simulation Loop ---
start_time = time.time()

while True:
    elapsed = time.time() - start_time
    if elapsed > TOTAL_DURATION:
        break
    
    # Calculate drift probability: P(drift) = t / T
    drift_ratio = elapsed / TOTAL_DURATION
    is_drift_sample = np.random.random() < drift_ratio
    
    # Sample selection
    if is_drift_sample and len(X_drift) > 0:
        idx = np.random.randint(0, len(X_drift), size=SAMPLES_PER_UPDATE)
        sample_x, actual_y = X_drift.iloc[idx], y_drift.iloc[idx]
    else:
        idx = np.random.randint(0, len(X_normal), size=SAMPLES_PER_UPDATE)
        sample_x, actual_y = X_normal.iloc[idx], y_normal.iloc[idx]

    # Endpoint Request
    pred_y_list = invoke_endpoint(sample_x)
    
    if pred_y_list is not None:
        for i, pred_y in enumerate(pred_y_list):
            history.append({'actual': actual_y.iloc[i], 'pred': int(pred_y)})
        
        # Calculate rolling metrics once we have enough data
        if len(history) >= 1:
            recent = history[-WINDOW_SIZE:]
            y_true = [i['actual'] for i in recent]
            y_pred = [i['pred'] for i in recent]
            
            # Record scores for plotting
            plot_times.append(elapsed)
            plot_precision.append(precision_score(y_true, y_pred, zero_division=0))
            plot_recall.append(recall_score(y_true, y_pred, zero_division=0))
            plot_f1.append(f1_score(y_true, y_pred, zero_division=0))
            plot_drift_pct.append(drift_ratio * 100)

            # --- Visualization ---
            display.clear_output(wait=True)
            fig, ax1 = plt.subplots(figsize=(12, 6))

            # Left Axis: Model Metrics
            ax1.plot(plot_times, plot_precision, label='Precision', color='#3498db', linewidth=2)
            ax1.plot(plot_times, plot_recall, label='Recall', color='#e74c3c', linewidth=2)
            ax1.plot(plot_times, plot_f1, label='F1-Score', color='#2ecc71', linewidth=3)
            ax1.set_ylabel('Score (0.0 - 1.0)')
            ax1.set_xlim(0, TOTAL_DURATION)
            ax1.set_ylim(0, 1.05)
            ax1.legend(loc='upper left')
            ax1.grid(axis='y', linestyle='--', alpha=0.7)

            # Right Axis: Drift Intensity
            ax2 = ax1.twinx()
            ax2.fill_between(plot_times, 0, plot_drift_pct, color='gray', alpha=0.1, label='Drift %')
            ax2.set_xlim(0, TOTAL_DURATION)
            ax2.set_ylabel('Data Drift Intensity (%)', color='gray')
            ax2.set_ylim(0, 100)

            plt.title(f"Live Model Decay Simulation\nDrift Intensity: {drift_ratio:.1%}")
            plt.xlabel("Time (seconds)")
            plt.show()

    time.sleep(0.1) # Frequency of requests

# Insurance

In [ ]:
import urllib.request
import json
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from IPython import display
from sklearn.metrics import precision_score, recall_score, f1_score

# --- Configuration ---
URL = '<your-endpoint-url>'
API_KEY = '<your-api-key>'
DATA_PATH = '../insurance/dataset/insurance.csv'
TOTAL_DURATION = 100 
WINDOW_SIZE = 100 # Number of recent samples for the "rolling" calculation
SAMPLES_PER_UPDATE = 10 # Number of samples per endpoint request

# --- Load and Prepare Data ---
df = pd.read_csv(DATA_PATH)
X_all = df.drop(['charges'], axis=1)
y_all = df['charges']

initial_regions = ['southwest', 'southeast']
mask_drift = df[~df['region'].isin(initial_regions)]
X_normal, y_normal = X_all[~mask_drift], y_all[~mask_drift]
X_drift, y_drift = X_all[mask_drift], y_all[mask_drift]

# --- Metrics Tracking ---
history = []      # Raw results: {'actual': y, 'pred': y_p}
plot_times = []
plot_rmse = []
plot_drift_pct = []

def invoke_endpoint(sample_rows):
    # Matches the required format: {"data": [[val1, val2, ...]]}
    payload = {"data": sample_rows.to_numpy().tolist()}
    body = str.encode(json.dumps(payload))
    headers = {'Content-Type': 'application/json', 'Authorization': f'Bearer {API_KEY}'}
    
    req = urllib.request.Request(URL, body, headers)
    try:
        with urllib.request.urlopen(req) as response:
            res = json.loads(response.read().decode("utf8"))
            # Extracts prediction from list or dict response
            return res if isinstance(res, list) else res.get('predict', [None])
    except Exception:
        return None

# --- Simulation Loop ---
start_time = time.time()

while True:
    elapsed = time.time() - start_time
    if elapsed > TOTAL_DURATION:
        break
    
    # Calculate drift probability: P(drift) = t / T
    # Rescale t[0]->10%, t[T]->50%
    drift_ratio = elapsed / TOTAL_DURATION
    drift_ratio = 0.1 + (0.4 * drift_ratio)  # Rescale to 10% to 50%
    is_drift_sample = np.random.random() < drift_ratio
    
    # Sample selection
    if is_drift_sample and len(X_drift) > 0:
        idx = np.random.randint(0, len(X_drift), size=SAMPLES_PER_UPDATE)
        sample_x, actual_y = X_drift.iloc[idx], y_drift.iloc[idx]
    else:
        idx = np.random.randint(0, len(X_normal), size=SAMPLES_PER_UPDATE)
        sample_x, actual_y = X_normal.iloc[idx], y_normal.iloc[idx]

    # Endpoint Request
    pred_y_list = invoke_endpoint(sample_x)
    
    if pred_y_list is not None:
        for i, pred_y in enumerate(pred_y_list):
            history.append({'actual': actual_y.iloc[i], 'pred': int(pred_y)})
        
        # Calculate rolling metrics once we have enough data
        if len(history) >= 1:
            recent = history[-WINDOW_SIZE:]
            y_true = [i['actual'] for i in recent]
            y_pred = [i['pred'] for i in recent]
            
            # Record scores for plotting
            plot_times.append(elapsed)
            plot_rmse.append(np.sqrt(((y_true - y_pred) ** 2).mean()))
            plot_drift_pct.append(drift_ratio * 100)

            # --- Visualization ---
            display.clear_output(wait=True)
            fig, ax1 = plt.subplots(figsize=(12, 6))

            # Left Axis: Model Metrics
            ax1.plot(plot_times, plot_rmse, label='RMSE', color='#3498db', linewidth=2)
            ax1.set_ylabel('RMSE')
            ax1.set_xlim(0, TOTAL_DURATION)
            ax1.set_ylim(0, max(plot_rmse) * 1.1)
            ax1.legend(loc='upper left')
            ax1.grid(axis='y', linestyle='--', alpha=0.7)

            # Right Axis: Drift Intensity
            ax2 = ax1.twinx()
            ax2.fill_between(plot_times, 0, plot_drift_pct, color='gray', alpha=0.1, label='Drift %')
            ax2.set_xlim(0, TOTAL_DURATION)
            ax2.set_ylabel('Data Drift Intensity (%)', color='gray')
            ax2.set_ylim(0, 100)

            plt.title(f"Live Model Decay Simulation\nDrift Intensity: {drift_ratio:.1%}")
            plt.xlabel("Time (seconds)")
            plt.show()

    time.sleep(0.1) # Frequency of requests